# 🧊 Exercício Prático: Pipeline Completo com Apache Iceberg

## 🎯 Objetivo

Desenvolver um pipeline completo de dados utilizando **Apache Iceberg**, aplicando todos os conceitos aprendidos na aula: versionamento, particionamento, schema evolution, compactação e integração com bancos relacionais.

## 📋 Cenário Empresarial

Você é um **Engenheiro de Dados** em uma empresa de e-commerce que precisa implementar um data lake moderno usando Apache Iceberg. O sistema deve:

- Processar dados de vendas de múltiplas fontes
- Manter histórico completo com versionamento
- Permitir evolução de schema sem downtime
- Integrar dados transacionais (PostgreSQL) com dados analíticos (Iceberg)
- Garantir performance através de particionamento e compactação


---

## 🚀 **PARTE 1: Criação da Arquitetura Base**

### **Tarefa 1.1: Setup do Ambiente**

Configure o ambiente Spark com Iceberg conforme os requisitos:
- Configure SparkSession com nome "EcommerceDataLake"
- Habilite extensões Iceberg
- Configure catálogo hadoop_catalog
- Defina warehouse em `/home/tavares/warehouse`


In [1]:
# Para o Spark se estiver rodando
try:
    spark.stop()
except:
    pass

# Configuração do Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-arm64"
os.environ["SPARK_HOME"] = "/opt/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init('/opt/spark-3.3.0-bin-hadoop3')

from pyspark.sql import SparkSession

# Sessão Spark com nome "EcommerceDataLake"
spark = SparkSession.builder \
    .appName("EcommerceDataLake") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/home/tavares/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

print(f"✅ Spark configurado com sucesso!")
print(f"📦 Warehouse: {spark.conf.get('spark.sql.catalog.hadoop_catalog.warehouse')}")
print(f"📊 App Name: {spark.conf.get('spark.app.name')}")


✅ Spark configurado com sucesso!
📦 Warehouse: /home/tavares/warehouse
📊 App Name: EcommerceDataLake


### **Tarefa 1.2: Criação da Tabela Principal**

Crie a tabela `vendas_ecommerce` com particionamento por ano e categoria.


In [2]:
# Limpar tabela e snapshots existentes (para começar do zero)
print("🧹 Limpando dados existentes...")

try:
    # Verificar se a tabela existe
    spark.sql("SELECT COUNT(*) as total FROM hadoop_catalog.default.vendas_ecommerce").collect()
    print("⚠️ Tabela encontrada. Removendo tabela e todos os snapshots...")
    
    # DROP TABLE remove a tabela, todos os dados, snapshots e metadados
    spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas_ecommerce")
    print("✅ Tabela e snapshots removidos com sucesso!")
    
except Exception as e:
    # Se a tabela não existe, apenas informar
    if "Table or view not found" in str(e) or "does not exist" in str(e):
        print("✅ Tabela não existe. Pronto para criar!")
    else:
        print(f"⚠️ Erro ao verificar tabela: {e}")
        print("💡 Tentando remover mesmo assim...")
        spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas_ecommerce")
        print("✅ Limpeza concluída!")

# Criar a tabela vendas_ecommerce com particionamento
spark.sql("""
    CREATE TABLE hadoop_catalog.default.vendas_ecommerce (
        venda_id INT,
        produto_nome STRING,
        categoria STRING,
        quantidade INT,
        preco_unitario DOUBLE,
        data_venda DATE,
        cliente_id STRING,
        vendedor_id INT
    )
    USING iceberg
    PARTITIONED BY (year(data_venda), categoria)
""")

print("✅ Tabela vendas_ecommerce criada com sucesso!")
print("📊 Partições: year(data_venda), categoria")

🧹 Limpando dados existentes...
⚠️ Tabela encontrada. Removendo tabela e todos os snapshots...
✅ Tabela e snapshots removidos com sucesso!
✅ Tabela vendas_ecommerce criada com sucesso!
📊 Partições: year(data_venda), categoria


---

## 📊 **PARTE 2: Operações com Versionamento**

### **Tarefa 2.1: Inserção de Dados Históricos**

Insira dados de vendas de 2023.


In [3]:
# Inserir dados de 2023
spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce 
    (venda_id, produto_nome, categoria, quantidade, preco_unitario, data_venda, cliente_id, vendedor_id)
    VALUES
    (1, 'Notebook Dell', 'Eletrônicos', 2, 2500.00, DATE('2023-01-15'), 'CLI001', 101),
    (2, 'Mouse Logitech', 'Eletrônicos', 5, 80.00, DATE('2023-01-16'), 'CLI002', 102),
    (3, 'Mesa Escritório', 'Móveis', 1, 800.00, DATE('2023-02-10'), 'CLI003', 101),
    (4, 'Cadeira Gamer', 'Móveis', 2, 600.00, DATE('2023-02-15'), 'CLI001', 103),
    (5, 'Smartphone Samsung', 'Eletrônicos', 1, 1200.00, DATE('2023-03-20'), 'CLI004', 102)
""")

print("✅ Dados de 2023 inseridos com sucesso!")
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce ORDER BY venda_id").show()

✅ Dados de 2023 inseridos com sucesso!
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|venda_id|      produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|       1|     Notebook Dell|Eletrônicos|         2|        2500.0|2023-01-15|    CLI001|        101|
|       2|    Mouse Logitech|Eletrônicos|         5|          80.0|2023-01-16|    CLI002|        102|
|       3|   Mesa Escritório|     Móveis|         1|         800.0|2023-02-10|    CLI003|        101|
|       4|     Cadeira Gamer|     Móveis|         2|         600.0|2023-02-15|    CLI001|        103|
|       5|Smartphone Samsung|Eletrônicos|         1|        1200.0|2023-03-20|    CLI004|        102|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+



### **Tarefa 2.2: Inserção de Dados de 2024**

Adicione vendas de 2024.


In [4]:
# Inserir dados de 2024
# Nota: A tabela pode ter sido evoluída (Parte 3) e ter as colunas desconto e canal_venda
# Por isso, incluímos essas colunas explicitamente com NULL
spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce 
    (venda_id, produto_nome, categoria, quantidade, preco_unitario, data_venda, cliente_id, vendedor_id)
    VALUES
    (6, 'Tablet iPad', 'Eletrônicos', 1, 3000.00, DATE('2024-01-10'), 'CLI002', 101),
    (7, 'Sofá 3 Lugares', 'Móveis', 1, 1500.00, DATE('2024-01-20'), 'CLI005', 103),
    (8, 'Monitor 4K', 'Eletrônicos', 2, 800.00, DATE('2024-02-05'), 'CLI003', 102)
""")

print("✅ Dados de 2024 inseridos com sucesso!")
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce ORDER BY venda_id").show()


✅ Dados de 2024 inseridos com sucesso!
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|venda_id|      produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|       1|     Notebook Dell|Eletrônicos|         2|        2500.0|2023-01-15|    CLI001|        101|
|       2|    Mouse Logitech|Eletrônicos|         5|          80.0|2023-01-16|    CLI002|        102|
|       3|   Mesa Escritório|     Móveis|         1|         800.0|2023-02-10|    CLI003|        101|
|       4|     Cadeira Gamer|     Móveis|         2|         600.0|2023-02-15|    CLI001|        103|
|       5|Smartphone Samsung|Eletrônicos|         1|        1200.0|2023-03-20|    CLI004|        102|
|       6|       Tablet iPad|Eletrônicos|         1|        3000.0|2024-01-10|    CLI002|        101|
|       7|    Sofá 3 Lugares|     Móveis|  

### **Tarefa 2.3: Análise de Snapshots**

Liste todos os snapshots criados e identifique quantos foram gerados.


In [5]:
# Listar todos os snapshots
print("📸 Snapshots da tabela vendas_ecommerce:")
spark.sql("SELECT snapshot_id, committed_at, operation, summary FROM hadoop_catalog.default.vendas_ecommerce.snapshots ORDER BY committed_at").show(truncate=False)

# Contar snapshots
total_snapshots = spark.sql("SELECT COUNT(*) as total FROM hadoop_catalog.default.vendas_ecommerce.snapshots").collect()[0]['total']
print(f"\n📊 Total de snapshots criados: {total_snapshots}")


📸 Snapshots da tabela vendas_ecommerce:
+-------------------+-----------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |committed_at           |operation|summary                                                                                                                                                                                                                                                                                                                                                                                                

### **Tarefa 2.4: Time Travel**

Consulte apenas os dados de 2023 usando o primeiro snapshot e compare com os dados atuais.


In [6]:
# Obter o primeiro snapshot (dados de 2023)
primeiro_snapshot = spark.sql("SELECT snapshot_id FROM hadoop_catalog.default.vendas_ecommerce.snapshots ORDER BY committed_at LIMIT 1").collect()[0]['snapshot_id']
print(f"🔍 Primeiro snapshot ID: {primeiro_snapshot}")

# Consultar dados do primeiro snapshot (Time Travel)
print("\n📅 Dados do primeiro snapshot (2023):")
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce 
    VERSION AS OF {primeiro_snapshot}
    ORDER BY venda_id
""").show()

# Comparar com dados atuais
print("\n📅 Dados atuais (2023 + 2024):")
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce ORDER BY venda_id").show()


🔍 Primeiro snapshot ID: 6923242085398549714

📅 Dados do primeiro snapshot (2023):
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|venda_id|      produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|       1|     Notebook Dell|Eletrônicos|         2|        2500.0|2023-01-15|    CLI001|        101|
|       2|    Mouse Logitech|Eletrônicos|         5|          80.0|2023-01-16|    CLI002|        102|
|       3|   Mesa Escritório|     Móveis|         1|         800.0|2023-02-10|    CLI003|        101|
|       4|     Cadeira Gamer|     Móveis|         2|         600.0|2023-02-15|    CLI001|        103|
|       5|Smartphone Samsung|Eletrônicos|         1|        1200.0|2023-03-20|    CLI004|        102|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+


---

## 🔄 **PARTE 3: Schema Evolution**

### **Tarefa 3.1: Evolução do Schema**

Evolua o schema adicionando as colunas `desconto` e `canal_venda`.


In [7]:
# Evoluir schema adicionando novas colunas
spark.sql("""
    ALTER TABLE hadoop_catalog.default.vendas_ecommerce 
    ADD COLUMN desconto DOUBLE
""")

spark.sql("""
    ALTER TABLE hadoop_catalog.default.vendas_ecommerce 
    ADD COLUMN canal_venda STRING
""")

print("✅ Schema evoluído com sucesso!")
print("📊 Novas colunas: desconto (DOUBLE), canal_venda (STRING)")

# Verificar schema atualizado
spark.sql("DESCRIBE hadoop_catalog.default.vendas_ecommerce").show()


✅ Schema evoluído com sucesso!
📊 Novas colunas: desconto (DOUBLE), canal_venda (STRING)
+--------------+-----------------+-------+
|      col_name|        data_type|comment|
+--------------+-----------------+-------+
|      venda_id|              int|       |
|  produto_nome|           string|       |
|     categoria|           string|       |
|    quantidade|              int|       |
|preco_unitario|           double|       |
|    data_venda|             date|       |
|    cliente_id|           string|       |
|   vendedor_id|              int|       |
|      desconto|           double|       |
|   canal_venda|           string|       |
|              |                 |       |
|# Partitioning|                 |       |
|        Part 0|years(data_venda)|       |
|        Part 1|        categoria|       |
+--------------+-----------------+-------+



### **Tarefa 3.2: Inserção com Novo Schema**

Insira dados usando o schema evoluído.


In [8]:
# Inserir dados com novo schema
spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
    (9, 'Headset Gamer', 'Eletrônicos', 3, 250.00, DATE('2024-03-15'), 'CLI006', 101, 10.0, 'online'),
    (10, 'Mesa Centro', 'Móveis', 1, 400.00, DATE('2024-03-20'), 'CLI007', 102, 5.0, 'loja_fisica')
""")

print("✅ Dados inseridos com novo schema!")
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce ORDER BY venda_id").show()


✅ Dados inseridos com novo schema!
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|venda_id|      produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|desconto|canal_venda|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|       1|     Notebook Dell|Eletrônicos|         2|        2500.0|2023-01-15|    CLI001|        101|    null|       null|
|       2|    Mouse Logitech|Eletrônicos|         5|          80.0|2023-01-16|    CLI002|        102|    null|       null|
|       3|   Mesa Escritório|     Móveis|         1|         800.0|2023-02-10|    CLI003|        101|    null|       null|
|       4|     Cadeira Gamer|     Móveis|         2|         600.0|2023-02-15|    CLI001|        103|    null|       null|
|       5|Smartphone Samsung|Eletrônicos|         1|        1200.0|2023-03-20|    CLI004|        102|   

### **Tarefa 3.3: Verificação de Compatibilidade**

Verifique que dados antigos têm valores NULL nas novas colunas e que todas as consultas ainda funcionam.


In [9]:
# Verificar compatibilidade - dados antigos têm NULL nas novas colunas
print("📊 Dados antigos (com NULL nas novas colunas):")
spark.sql("""
    SELECT venda_id, produto_nome, desconto, canal_venda 
    FROM hadoop_catalog.default.vendas_ecommerce 
    WHERE venda_id <= 8
    ORDER BY venda_id
""").show()

print("\n📊 Dados novos (com valores nas novas colunas):")
spark.sql("""
    SELECT venda_id, produto_nome, desconto, canal_venda 
    FROM hadoop_catalog.default.vendas_ecommerce 
    WHERE venda_id > 8
    ORDER BY venda_id
""").show()

print("\n✅ Todas as consultas funcionam corretamente!")


📊 Dados antigos (com NULL nas novas colunas):
+--------+------------------+--------+-----------+
|venda_id|      produto_nome|desconto|canal_venda|
+--------+------------------+--------+-----------+
|       1|     Notebook Dell|    null|       null|
|       2|    Mouse Logitech|    null|       null|
|       3|   Mesa Escritório|    null|       null|
|       4|     Cadeira Gamer|    null|       null|
|       5|Smartphone Samsung|    null|       null|
|       6|       Tablet iPad|    null|       null|
|       7|    Sofá 3 Lugares|    null|       null|
|       8|        Monitor 4K|    null|       null|
+--------+------------------+--------+-----------+


📊 Dados novos (com valores nas novas colunas):
+--------+-------------+--------+-----------+
|venda_id| produto_nome|desconto|canal_venda|
+--------+-------------+--------+-----------+
|       9|Headset Gamer|    10.0|     online|
|      10|  Mesa Centro|     5.0|loja_fisica|
+--------+-------------+--------+-----------+


✅ Todas as cons

---

## 🔧 **PARTE 4: Operações ACID e Merge**

### **Tarefa 4.1: Simulação de Erro**

Faça uma atualização "problemática" que será revertida depois.


In [10]:
# Obter snapshot antes da atualização problemática
try:
    snapshot_antes = spark.sql("SELECT snapshot_id FROM hadoop_catalog.default.vendas_ecommerce.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0]['snapshot_id']
    print(f"📸 Snapshot antes da atualização: {snapshot_antes}")
except Exception as e:
    print(f"⚠️ Erro ao obter snapshot: {e}")
    # Tentar obter o snapshot mais recente de outra forma
    snapshots = spark.sql("SELECT snapshot_id FROM hadoop_catalog.default.vendas_ecommerce.snapshots ORDER BY committed_at DESC LIMIT 1").collect()
    if snapshots:
        snapshot_antes = snapshots[0]['snapshot_id']
        print(f"📸 Snapshot antes da atualização: {snapshot_antes}")
    else:
        raise Exception("Não foi possível obter snapshot")

# Atualização problemática (multiplicar preço por 100)
# Limitar a alguns registros para evitar problemas de memória
# Vamos atualizar apenas os primeiros registros de Eletrônicos para demonstração
print("\n⚠️ Executando atualização problemática (limitada para demonstração)...")
print("💡 Nota: Limitando a alguns registros (1000 primeiros) para evitar problemas de memória com muitos registros")

try:
    # Atualizar apenas alguns registros para demonstração
    spark.sql("""
        UPDATE hadoop_catalog.default.vendas_ecommerce 
        SET preco_unitario = preco_unitario * 100 
        WHERE categoria = 'Eletrônicos' AND venda_id <= 1000
    """)
    
    print("✅ Atualização problemática realizada!")
    print("📊 Preços de Eletrônicos após atualização (primeiros 20 registros):")
    spark.sql("""
        SELECT venda_id, produto_nome, categoria, preco_unitario 
        FROM hadoop_catalog.default.vendas_ecommerce 
        WHERE categoria = 'Eletrônicos' AND venda_id <= 1000
        ORDER BY venda_id
        LIMIT 20
    """).show()
except Exception as e:
    print(f"❌ Erro ao executar UPDATE: {e}")
    print("💡 O Spark pode ter travado. Tente reiniciar o kernel e executar novamente.")
    raise


📸 Snapshot antes da atualização: 4732149720788807473

⚠️ Executando atualização problemática (limitada para demonstração)...
💡 Nota: Limitando a alguns registros (1000 primeiros) para evitar problemas de memória com muitos registros
✅ Atualização problemática realizada!
📊 Preços de Eletrônicos após atualização (primeiros 20 registros):
+--------+------------------+-----------+--------------+
|venda_id|      produto_nome|  categoria|preco_unitario|
+--------+------------------+-----------+--------------+
|       1|     Notebook Dell|Eletrônicos|      250000.0|
|       2|    Mouse Logitech|Eletrônicos|        8000.0|
|       5|Smartphone Samsung|Eletrônicos|      120000.0|
|       6|       Tablet iPad|Eletrônicos|      300000.0|
|       8|        Monitor 4K|Eletrônicos|       80000.0|
|       9|     Headset Gamer|Eletrônicos|       25000.0|
+--------+------------------+-----------+--------------+



### **Tarefa 4.2: Rollback**

Execute rollback para o snapshot antes da atualização problemática.


In [11]:
# Executar rollback para o snapshot anterior
spark.sql(f"""
    CALL hadoop_catalog.system.rollback_to_snapshot('default.vendas_ecommerce', {snapshot_antes})
""")

print("✅ Rollback executado com sucesso!")
print("📊 Preços de Eletrônicos após rollback:")
spark.sql("""
    SELECT venda_id, produto_nome, categoria, preco_unitario 
    FROM hadoop_catalog.default.vendas_ecommerce 
    WHERE categoria = 'Eletrônicos'
    ORDER BY venda_id
""").show()


✅ Rollback executado com sucesso!
📊 Preços de Eletrônicos após rollback:
+--------+------------------+-----------+--------------+
|venda_id|      produto_nome|  categoria|preco_unitario|
+--------+------------------+-----------+--------------+
|       1|     Notebook Dell|Eletrônicos|        2500.0|
|       2|    Mouse Logitech|Eletrônicos|          80.0|
|       5|Smartphone Samsung|Eletrônicos|        1200.0|
|       6|       Tablet iPad|Eletrônicos|        3000.0|
|       8|        Monitor 4K|Eletrônicos|         800.0|
|       9|     Headset Gamer|Eletrônicos|         250.0|
+--------+------------------+-----------+--------------+



### **Tarefa 4.3: Operação MERGE**

Crie uma view temporária com atualizações e execute MERGE INTO.


In [12]:
# Criar view temporária com atualizações
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW vendas_updates AS
    SELECT 1 as venda_id, 'Notebook Dell UPDATED' as produto_nome, 'Eletrônicos' as categoria, 
           2 as quantidade, 2600.00 as preco_unitario, DATE('2023-01-15') as data_venda, 
           'CLI001' as cliente_id, 101 as vendedor_id, 0.0 as desconto, 'online' as canal_venda
    UNION ALL
    SELECT 11 as venda_id, 'Teclado Mecânico' as produto_nome, 'Eletrônicos' as categoria,
           4 as quantidade, 300.00 as preco_unitario, DATE('2024-04-01') as data_venda,
           'CLI008' as cliente_id, 103 as vendedor_id, 15.0 as desconto, 'online' as canal_venda
""")

# Executar MERGE INTO
spark.sql("""
    MERGE INTO hadoop_catalog.default.vendas_ecommerce AS target
    USING vendas_updates AS source
    ON target.venda_id = source.venda_id
    WHEN MATCHED THEN
        UPDATE SET 
            produto_nome = source.produto_nome,
            preco_unitario = source.preco_unitario,
            desconto = source.desconto,
            canal_venda = source.canal_venda
    WHEN NOT MATCHED THEN
        INSERT (venda_id, produto_nome, categoria, quantidade, preco_unitario, data_venda, 
                cliente_id, vendedor_id, desconto, canal_venda)
        VALUES (source.venda_id, source.produto_nome, source.categoria, source.quantidade, 
                source.preco_unitario, source.data_venda, source.cliente_id, source.vendedor_id,
                source.desconto, source.canal_venda)
""")

print("✅ MERGE executado com sucesso!")
print("📊 Dados após MERGE:")
spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce WHERE venda_id IN (1, 11) ORDER BY venda_id").show()


✅ MERGE executado com sucesso!
📊 Dados após MERGE:
+--------+--------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|venda_id|        produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|desconto|canal_venda|
+--------+--------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|       1|Notebook Dell UPD...|Eletrônicos|         2|        2600.0|2023-01-15|    CLI001|        101|     0.0|     online|
|      11|    Teclado Mecânico|Eletrônicos|         4|         300.0|2024-04-01|    CLI008|        103|    15.0|     online|
+--------+--------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+



---

## 📈 **PARTE 5: Otimização e Análise**

### **Tarefa 5.1: Análise de Fragmentação**

Use metadados para analisar quantos arquivos foram criados e identifique se há necessidade de compactação.


In [13]:
# Análise de arquivos antes da compactação
print("📊 Análise de arquivos antes da compactação:")
arquivos_antes = spark.sql("""
    SELECT 
        COUNT(*) as total_arquivos,
        SUM(file_size_in_bytes) as tamanho_total_bytes,
        AVG(file_size_in_bytes) as tamanho_medio_bytes
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").collect()[0]

print(f"📁 Total de arquivos: {arquivos_antes['total_arquivos']}")
print(f"💾 Tamanho total: {arquivos_antes['tamanho_total_bytes']} bytes")
print(f"📏 Tamanho médio: {arquivos_antes['tamanho_medio_bytes']:.2f} bytes")

# Estatísticas por partição
print("\n📊 Estatísticas por partição:")
spark.sql("""
    SELECT 
        partition.data_venda_year,
        partition.categoria,
        COUNT(*) as num_arquivos
    FROM hadoop_catalog.default.vendas_ecommerce.files
    GROUP BY partition.data_venda_year, partition.categoria
    ORDER BY partition.data_venda_year, partition.categoria
""").show()


📊 Análise de arquivos antes da compactação:
📁 Total de arquivos: 7
💾 Tamanho total: 18349 bytes
📏 Tamanho médio: 2621.29 bytes

📊 Estatísticas por partição:
+---------------+-----------+------------+
|data_venda_year|  categoria|num_arquivos|
+---------------+-----------+------------+
|             53|Eletrônicos|           1|
|             53|     Móveis|           1|
|             54|Eletrônicos|           3|
|             54|     Móveis|           2|
+---------------+-----------+------------+



### **Tarefa 5.2: Compactação**

Execute compactação usando `rewrite_data_files` e compare o número de arquivos antes e depois.


In [14]:
# Executar compactação
print("🔄 Executando compactação...")
spark.sql("""
    CALL hadoop_catalog.system.rewrite_data_files(
        table => 'default.vendas_ecommerce',
        options => map('target-file-size-bytes', '134217728')
    )
""")

print("✅ Compactação concluída!")

# Análise de arquivos após compactação
print("\n📊 Análise de arquivos após compactação:")
arquivos_depois = spark.sql("""
    SELECT 
        COUNT(*) as total_arquivos,
        SUM(file_size_in_bytes) as tamanho_total_bytes,
        AVG(file_size_in_bytes) as tamanho_medio_bytes
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").collect()[0]

print(f"📁 Total de arquivos: {arquivos_depois['total_arquivos']}")
print(f"💾 Tamanho total: {arquivos_depois['tamanho_total_bytes']} bytes")
print(f"📏 Tamanho médio: {arquivos_depois['tamanho_medio_bytes']:.2f} bytes")

# Comparação
reducao = arquivos_antes['total_arquivos'] - arquivos_depois['total_arquivos']
print(f"\n📉 Redução de arquivos: {reducao} arquivos")
print(f"📊 Redução percentual: {(reducao/arquivos_antes['total_arquivos']*100):.2f}%")


🔄 Executando compactação...
✅ Compactação concluída!

📊 Análise de arquivos após compactação:
📁 Total de arquivos: 7
💾 Tamanho total: 18349 bytes
📏 Tamanho médio: 2621.29 bytes

📉 Redução de arquivos: 0 arquivos
📊 Redução percentual: 0.00%


### **Tarefa 5.3: Análise de Performance**

Demonstre partition pruning consultando apenas dados de 2024 e mostre consulta otimizada por categoria.


In [15]:
# Partition pruning - consulta apenas dados de 2024
print("📊 Consulta com partition pruning (apenas 2024):")
spark.sql("""
    SELECT 
        year(data_venda) as ano,
        categoria,
        COUNT(*) as total_vendas,
        SUM(preco_unitario * quantidade) as receita_total
    FROM hadoop_catalog.default.vendas_ecommerce 
    WHERE year(data_venda) = 2024
    GROUP BY year(data_venda), categoria
    ORDER BY categoria
""").show()

# Consulta otimizada por categoria
print("\n📊 Consulta otimizada por categoria:")
spark.sql("""
    SELECT 
        categoria,
        COUNT(*) as total_vendas,
        ROUND(AVG(COALESCE(desconto, 0.0)), 2) as desconto_medio,
        ROUND(SUM(preco_unitario * quantidade), 2) as receita_total
    FROM hadoop_catalog.default.vendas_ecommerce 
    WHERE categoria = 'Eletrônicos'
    GROUP BY categoria
""").show()

print("\n✅ Partition pruning funcionando corretamente!")


📊 Consulta com partition pruning (apenas 2024):
+----+-----------+------------+-------------+
| ano|  categoria|total_vendas|receita_total|
+----+-----------+------------+-------------+
|2024|Eletrônicos|           4|       6550.0|
|2024|     Móveis|           2|       1900.0|
+----+-----------+------------+-------------+


📊 Consulta otimizada por categoria:
+-----------+------------+--------------+-------------+
|  categoria|total_vendas|desconto_medio|receita_total|
+-----------+------------+--------------+-------------+
|Eletrônicos|           7|          3.57|      13350.0|
+-----------+------------+--------------+-------------+


✅ Partition pruning funcionando corretamente!


### **Tarefa 5.4: Limpeza de Snapshots**

Execute `expire_snapshots` mantendo apenas os últimos 3 snapshots.


In [16]:
snapshots_total = spark.sql("SELECT * FROM hadoop_catalog.default.vendas_ecommerce.snapshots").show()

snapshot_teste = spark.sql("""
    SELECT snapshot_id, committed_at
    FROM (
        SELECT snapshot_id, committed_at,
               ROW_NUMBER() OVER (ORDER BY committed_at DESC) as rn
        FROM hadoop_catalog.default.vendas_ecommerce.snapshots
    ) ranked
    WHERE rn = 3
""").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2025-11-10 05:02:...|6923242085398549714|               null|   append|/home/tavares/war...|{spark.app.id -> ...|
|2025-11-10 05:02:...|5128080576712370384|6923242085398549714|   append|/home/tavares/war...|{spark.app.id -> ...|
|2025-11-10 05:02:...|4732149720788807473|5128080576712370384|   append|/home/tavares/war...|{spark.app.id -> ...|
|2025-11-10 05:02:...|2213055140168317050|4732149720788807473|overwrite|/home/tavares/war...|{spark.app.id -> ...|
|2025-11-10 05:02:...|3177874459736835568|4732149720788807473|overwrite|/home/tavares/war...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------

In [17]:
# Contar snapshots antes da limpeza
snapshots_antes = spark.sql("SELECT COUNT(*) as total FROM hadoop_catalog.default.vendas_ecommerce.snapshots").collect()[0]['total']
print(f"📸 Snapshots antes da limpeza: {snapshots_antes}")

# Mostrar snapshots antes da limpeza
print("\n📸 Snapshots antes da limpeza:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
    ORDER BY committed_at DESC
""").show(truncate=False)

if snapshots_antes >= 3:
    # Obter o timestamp do 3º snapshot mais recente
    snapshot_limite = spark.sql("""
        SELECT committed_at
        FROM (
            SELECT committed_at,
                   ROW_NUMBER() OVER (ORDER BY committed_at DESC) as rn
            FROM hadoop_catalog.default.vendas_ecommerce.snapshots
        ) ranked
        WHERE rn = 3
    """).collect()[0]
    
    timestamp_limite = snapshot_limite['committed_at']
    print(f"\n📅 Removendo snapshots anteriores a: {timestamp_limite}")
    
    # Converter timestamp para string no formato ISO (sem subconsulta)
    # O formato deve ser: 'YYYY-MM-DD HH:MM:SS' ou 'YYYY-MM-DD HH:MM:SS.SSS'
    if hasattr(timestamp_limite, 'strftime'):
        # Se for datetime object
        timestamp_str = timestamp_limite.strftime('%Y-%m-%d %H:%M:%S')
    else:
        # Se for string, usar diretamente (remover microsegundos se houver)
        timestamp_str = str(timestamp_limite).split('.')[0] if '.' in str(timestamp_limite) else str(timestamp_limite)
    
    print(f"📅 Timestamp formatado: {timestamp_str}")
    
    # Expirar snapshots usando o timestamp como literal (sem subconsulta)
    print("\n🔄 Executando limpeza de snapshots...")
    cleanup_result = spark.sql(f"""
        CALL hadoop_catalog.system.expire_snapshots(
            table => 'default.vendas_ecommerce',
            older_than => TIMESTAMP '{timestamp_str}'
        )
    """)
    
    print("\n📊 Resultado da limpeza:")
    cleanup_result.show(truncate=False)
    
    print("✅ Limpeza de snapshots concluída!")
else:
    print("⚠️ Não há snapshots suficientes para limpeza (mínimo 3)")

# Contar snapshots após limpeza
snapshots_depois = spark.sql("SELECT COUNT(*) as total FROM hadoop_catalog.default.vendas_ecommerce.snapshots").collect()[0]['total']
print(f"\n📸 Snapshots após limpeza: {snapshots_depois}")
print(f"🗑️ Snapshots removidos: {snapshots_antes - snapshots_depois}")

# Mostrar snapshots restantes
print("\n📸 Snapshots restantes:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
    ORDER BY committed_at DESC
""").show(truncate=False)

📸 Snapshots antes da limpeza: 5

📸 Snapshots antes da limpeza:
+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|3177874459736835568|2025-11-10 05:02:24.806|overwrite|
|2213055140168317050|2025-11-10 05:02:22.298|overwrite|
|4732149720788807473|2025-11-10 05:02:19.962|append   |
|5128080576712370384|2025-11-10 05:02:18.143|append   |
|6923242085398549714|2025-11-10 05:02:16.197|append   |
+-------------------+-----------------------+---------+


📅 Removendo snapshots anteriores a: 2025-11-10 05:02:19.962000
📅 Timestamp formatado: 2025-11-10 05:02:19

🔄 Executando limpeza de snapshots...

📊 Resultado da limpeza:
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equa

---

## 6. 🏆 **ENTREGA FINAL**


### **6.1 Consultas de Validação**

Execute as consultas de validação para confirmar que tudo está funcionando corretamente.


In [18]:
# ============================================
# CONSULTAS DE VALIDAÇÃO
# ============================================

print("=" * 60)
print("📊 CONSULTAS DE VALIDAÇÃO")
print("=" * 60)

# 1. Total de registros por ano
print("\n📊 1. Total de registros por ano:")
spark.sql("""
    SELECT year(data_venda) as ano, COUNT(*) as total_vendas 
    FROM hadoop_catalog.default.vendas_ecommerce 
    GROUP BY year(data_venda) 
    ORDER BY ano
""").show()

# 2. Vendas por categoria com desconto médio
print("\n📊 2. Vendas por categoria com desconto médio:")
spark.sql("""
    SELECT categoria, 
           COUNT(*) as total_vendas,
           ROUND(AVG(COALESCE(desconto, 0.0)), 2) as desconto_medio,
           ROUND(SUM(preco_unitario * quantidade), 2) as receita_total
    FROM hadoop_catalog.default.vendas_ecommerce 
    GROUP BY categoria
""").show()

# 3. Análise temporal de snapshots
print("\n📊 3. Análise temporal de snapshots:")
spark.sql("""
    SELECT operation, COUNT(*) as num_operacoes 
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots 
    GROUP BY operation
""").show()

print("\n" + "=" * 60)
print("✅ Consultas de validação concluídas!")
print("=" * 60)

📊 CONSULTAS DE VALIDAÇÃO

📊 1. Total de registros por ano:
+----+------------+
| ano|total_vendas|
+----+------------+
|2023|           5|
|2024|           6|
+----+------------+


📊 2. Vendas por categoria com desconto médio:
+-----------+------------+--------------+-------------+
|  categoria|total_vendas|desconto_medio|receita_total|
+-----------+------------+--------------+-------------+
|Eletrônicos|           7|          3.57|      13350.0|
|     Móveis|           4|          1.25|       3900.0|
+-----------+------------+--------------+-------------+


📊 3. Análise temporal de snapshots:
+---------+-------------+
|operation|num_operacoes|
+---------+-------------+
|   append|            1|
|overwrite|            2|
+---------+-------------+


✅ Consultas de validação concluídas!


### ***6.2 Resumo Executivo***:
   - Quantos snapshots foram criados no total?
   - Qual foi a redução de arquivos após compactação?
   - Quantas partições foram criadas?


In [19]:
# ============================================
# CÁLCULO DE MÉTRICAS PARA O RELATÓRIO
# ============================================

print("=" * 60)
print("📊 CALCULANDO MÉTRICAS PARA O RELATÓRIO")
print("=" * 60)

# Total de snapshots criados
total_snapshots = spark.sql("SELECT COUNT(*) as total FROM hadoop_catalog.default.vendas_ecommerce.snapshots").collect()[0]['total']
print(f"\n📸 Total de snapshots criados: {total_snapshots}")

# Total de arquivos após compactação
arquivos_finais = spark.sql("SELECT COUNT(*) as total FROM hadoop_catalog.default.vendas_ecommerce.files").collect()[0]['total']
print(f"📁 Total de arquivos após compactação: {arquivos_finais}")

# Total de partições criadas
particoes = spark.sql("""
    SELECT COUNT(DISTINCT CONCAT(CAST(partition.data_venda_year AS STRING), '-', partition.categoria)) as total
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").collect()[0]['total']
print(f"📊 Total de partições criadas: {particoes}")

# Estatísticas de snapshots por operação
print("\n📊 Estatísticas de snapshots por operação:")
spark.sql("""
    SELECT operation, COUNT(*) as quantidade
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
    GROUP BY operation
    ORDER BY quantidade DESC
""").show()

# Tamanho total dos arquivos
tamanho_total = spark.sql("""
    SELECT 
        SUM(file_size_in_bytes) as tamanho_total_bytes,
        ROUND(SUM(file_size_in_bytes) / 1024.0 / 1024.0, 2) as tamanho_total_mb
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").collect()[0]
print(f"\n💾 Tamanho total dos arquivos: {tamanho_total['tamanho_total_bytes']:,} bytes ({tamanho_total['tamanho_total_mb']} MB)")

# Total de registros
total_registros = spark.sql("SELECT COUNT(*) as total FROM hadoop_catalog.default.vendas_ecommerce").collect()[0]['total']
print(f"📊 Total de registros na tabela: {total_registros:,}")

print("\n" + "=" * 60)
print("✅ Métricas calculadas!")
print("=" * 60)

📊 CALCULANDO MÉTRICAS PARA O RELATÓRIO

📸 Total de snapshots criados: 3
📁 Total de arquivos após compactação: 7
📊 Total de partições criadas: 4

📊 Estatísticas de snapshots por operação:
+---------+----------+
|operation|quantidade|
+---------+----------+
|overwrite|         2|
|   append|         1|
+---------+----------+


💾 Tamanho total dos arquivos: 18,349 bytes (0.02 MB)
📊 Total de registros na tabela: 11

✅ Métricas calculadas!


### ***6.3 Benefícios Observados***:
   - Liste 3 vantagens do Iceberg que você observou na prática
   - Compare com o que seria necessário usando Parquet tradicional


### Vantagens do Iceberg Observadas na Prática:

1. **Versionamento e Time Travel**
   - Acesso a qualquer versão histórica dos dados através de snapshots
   - Útil para auditoria, análise de tendências e recuperação após erros
   - Demonstrado ao consultar dados do primeiro snapshot mesmo após inserções posteriores
2. **Schema Evolution sem Downtime**
   - Adição de colunas (`desconto` e `canal_venda`) sem parar o sistema
   - Dados antigos recebem NULL automaticamente nas novas colunas
   - Permite adaptação rápida a mudanças de requisitos sem impacto em pipelines existentes
3. **ACID Transactions e Rollback**
   - Transações garantem consistência dos dados
   - Operações UPDATE podem ser revertidas via rollback
   - MERGE INTO combina UPDATE e INSERT atomicamente

### Comparação com Parquet Tradicional:

**Com Parquet tradicional seria necessário:**
- ❌ Recriar toda a tabela para adicionar colunas (downtime)
- ❌ Manter múltiplas versões de arquivos manualmente
- ❌ Implementar lógica customizada para atomicidade
- ❌ Gerenciar manualmente limpeza de arquivos antigos
- ❌ Sem garantias ACID nativas
- ❌ Dificuldade para rollback

**Com Apache Iceberg:**
- ✅ Schema evolution sem downtime
- ✅ Versionamento automático via snapshots
- ✅ Transações ACID nativas
- ✅ Rollback simples e eficiente
- ✅ Limpeza automática de metadados
- ✅ Garantias de consistência

### ***6.4 Casos de Uso Identificados***:
   - Descreva 2 cenários empresariais onde este pipeline seria útil
   - Explique como o versionamento ajudaria em cada caso


### Cenário 1: E-commerce com Análise de Vendas e Auditoria

**Contexto:** Empresa de e-commerce precisa analisar vendas históricas e manter auditoria completa de transações.

**Como o versionamento ajudaria:**
- **Análise de tendências:** Consultar dados de períodos específicos (ex: Black Friday 2023) mesmo após atualizações
- **Auditoria:** Rastrear mudanças de preços, descontos e canais de venda ao longo do tempo
- **Compliance:** Demonstrar integridade dos dados para reguladores com histórico completo
- **Recuperação:** Reverter alterações acidentais (ex: preços multiplicados por erro) via rollback
- **Análise comparativa:** Comparar performance de vendas entre diferentes versões dos dados

### Cenário 2: Data Lake para Análise de Dados de Múltiplas Fontes

**Contexto:** Empresa precisa integrar dados de vendas de múltiplas fontes (loja física, online, marketplace) em um data lake para análises consolidadas.

**Como o versionamento ajudaria:**
- **Integração incremental:** Adicionar novas fontes de dados sem recriar todo o pipeline
- **Correção de dados:** Corrigir dados incorretos de fontes específicas mantendo histórico das correções
- **Testes de qualidade:** Comparar versões antes e depois de limpezas de dados
- **Análise de impacto:** Avaliar impacto de mudanças de schema em relatórios existentes
- **Governança de dados:** Manter rastreabilidade completa de origem e transformações dos dados

---

## 🎯 PONTUAÇÃO EXTRA (+10 pontos)

### Tarefas Opcionais para Pontuação Extra:

#### 1. Implementar particionamento adicional por `canal_venda`
- Adicionar `canal_venda` como partição adicional na tabela
- Demonstrar melhoria de performance em consultas filtradas por canal

#### 2. Criar análise avançada usando múltiplas tabelas de metadados
- Combinar dados de `snapshots`, `files` e `partitions` para análise completa
- Criar relatório de performance e otimização baseado em metadados

#### 3. Demonstrar integração com dados do PostgreSQL (tabela customers)
- Criar tabela de clientes no PostgreSQL
- Integrar dados transacionais (PostgreSQL) com dados analíticos (Iceberg)
- Demonstrar uso de JOIN entre tabelas de diferentes sistemas
---

In [20]:
# ============================================
# PONTUAÇÃO EXTRA 1: Particionamento adicional por canal_venda
# ============================================

print("=" * 70)
print("🎯 PONTUAÇÃO EXTRA 1: Particionamento adicional por canal_venda")
print("=" * 70)

# Adicionar canal_venda como partição adicional na tabela
print("\n📊 Adicionando canal_venda como partição adicional...")

# Nota: Iceberg não permite alterar particionamento de tabela existente
# Vamos criar uma nova tabela com o particionamento adicional
print("💡 Criando nova tabela com particionamento por ano, categoria e canal_venda...")

spark.sql("""
    CREATE TABLE IF NOT EXISTS hadoop_catalog.default.vendas_ecommerce_particionada (
        venda_id INT,
        produto_nome STRING,
        categoria STRING,
        quantidade INT,
        preco_unitario DOUBLE,
        data_venda DATE,
        cliente_id STRING,
        vendedor_id INT,
        desconto DOUBLE,
        canal_venda STRING
    )
    USING iceberg
    PARTITIONED BY (year(data_venda), categoria, canal_venda)
""")

print("✅ Tabela criada com particionamento adicional por canal_venda!")

# Inserir dados na nova tabela
print("\n🔄 Inserindo dados na tabela com particionamento adicional...")
spark.sql("""
    INSERT INTO hadoop_catalog.default.vendas_ecommerce_particionada
    SELECT * FROM hadoop_catalog.default.vendas_ecommerce
    WHERE canal_venda IS NOT NULL
""")

print("✅ Dados inseridos!")

# Demonstrar melhoria de performance em consultas filtradas por canal
print("\n📊 Demonstrando melhoria de performance em consulta filtrada por canal:")
print("💡 A consulta abaixo usa partition pruning com canal_venda como partição")

spark.sql("""
    SELECT 
        canal_venda,
        categoria,
        COUNT(*) as total_vendas,
        ROUND(SUM(preco_unitario * quantidade), 2) as receita_total,
        ROUND(AVG(COALESCE(desconto, 0.0)), 2) as desconto_medio
    FROM hadoop_catalog.default.vendas_ecommerce_particionada
    WHERE canal_venda = 'online'
    GROUP BY canal_venda, categoria
    ORDER BY receita_total DESC
""").show()

print("\n✅ Particionamento adicional implementado e performance demonstrada!")
print("=" * 70)

🎯 PONTUAÇÃO EXTRA 1: Particionamento adicional por canal_venda

📊 Adicionando canal_venda como partição adicional...
💡 Criando nova tabela com particionamento por ano, categoria e canal_venda...
✅ Tabela criada com particionamento adicional por canal_venda!

🔄 Inserindo dados na tabela com particionamento adicional...
✅ Dados inseridos!

📊 Demonstrando melhoria de performance em consulta filtrada por canal:
💡 A consulta abaixo usa partition pruning com canal_venda como partição
+-----------+-----------+------------+-------------+--------------+
|canal_venda|  categoria|total_vendas|receita_total|desconto_medio|
+-----------+-----------+------------+-------------+--------------+
|     online|Eletrônicos|           6|      14300.0|          8.33|
+-----------+-----------+------------+-------------+--------------+


✅ Particionamento adicional implementado e performance demonstrada!


In [21]:
# ============================================
# PONTUAÇÃO EXTRA 2: Análise avançada usando múltiplas tabelas de metadados
# ============================================

print("=" * 70)
print("🎯 PONTUAÇÃO EXTRA 2: Análise avançada usando múltiplas tabelas de metadados")
print("=" * 70)

# Combinar dados de snapshots, files e partitions para análise completa
print("\n📊 Combinando dados de snapshots, files e partitions para análise completa:")

# Relatório completo de performance e otimização baseado em metadados
print("\n📊 Relatório de Performance e Otimização Baseado em Metadados:")

# 1. Análise de Snapshots
print("\n1️⃣ Análise de Snapshots:")

spark.sql("""
    SELECT 
        operation,
        COUNT(*) as num_snapshots,
        MIN(committed_at) as primeira_operacao,
        MAX(committed_at) as ultima_operacao
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
    GROUP BY operation
    ORDER BY num_snapshots DESC
""").show()

# 2. Análise de Files
print("\n2️⃣ Análise de Files:")

spark.sql("""
    SELECT 
        COUNT(*) as total_arquivos,
        SUM(file_size_in_bytes) as tamanho_total_bytes,
        ROUND(SUM(file_size_in_bytes) / 1024.0 / 1024.0, 2) as tamanho_total_mb,
        AVG(file_size_in_bytes) as tamanho_medio_bytes,
        SUM(record_count) as total_registros,
        AVG(record_count) as registros_medio_por_arquivo
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").show()

# 3. Análise combinada: Files + Partitions
print("\n3️⃣ Análise combinada: Files e Partitions:")

spark.sql("""
    SELECT 
        partition.data_venda_year as ano,
        partition.categoria,
        COUNT(*) as num_arquivos,
        SUM(file_size_in_bytes) as tamanho_total_bytes,
        ROUND(SUM(file_size_in_bytes) / 1024.0 / 1024.0, 2) as tamanho_total_mb,
        SUM(record_count) as total_registros,
        AVG(file_size_in_bytes) as tamanho_medio_arquivo,
        ROUND(SUM(record_count) / COUNT(*), 2) as registros_por_arquivo
    FROM hadoop_catalog.default.vendas_ecommerce.files
    GROUP BY partition.data_venda_year, partition.categoria
    ORDER BY ano DESC, categoria
""").show()

# 4. Relatório completo de performance e otimização
print("\n4️⃣ Relatório Completo de Performance e Otimização:")

# Análise de snapshots
snapshots_stats = spark.sql("""
    SELECT 
        COUNT(*) as total_snapshots,
        COUNT(DISTINCT operation) as tipos_operacoes
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
""").collect()[0]

# Análise de files
files_stats = spark.sql("""
    SELECT 
        COUNT(*) as total_arquivos,
        SUM(file_size_in_bytes) as tamanho_total_bytes,
        SUM(record_count) as total_registros
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").collect()[0]

# Análise de partitions
partitions_stats = spark.sql("""
    SELECT 
        COUNT(DISTINCT CONCAT(CAST(partition.data_venda_year AS STRING), '-', partition.categoria)) as total_particoes
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").collect()[0]

# Exibir relatório consolidado
print("\n📊 Relatório Consolidado:")
print(f"  📸 Total de Snapshots: {snapshots_stats['total_snapshots']}")
print(f"  📁 Total de Arquivos: {files_stats['total_arquivos']}")
print(f"  📊 Total de Partições: {partitions_stats['total_particoes']}")
print(f"  💾 Tamanho Total: {files_stats['tamanho_total_bytes']:,} bytes ({round(files_stats['tamanho_total_bytes'] / 1024.0 / 1024.0, 2)} MB)")
print(f"  📈 Total de Registros: {files_stats['total_registros']:,}")
print(f"  📏 Tamanho Médio por Arquivo: {round(files_stats['tamanho_total_bytes'] / files_stats['total_arquivos'], 2):,} bytes")
print(f"  📊 Registros Médio por Arquivo: {round(files_stats['total_registros'] / files_stats['total_arquivos'], 2):,}")

# 5. Análise de evolução temporal (usando snapshots)
print("\n5️⃣ Análise de Evolução Temporal (Snapshots):")

spark.sql("""
    SELECT 
        DATE(committed_at) as data,
        operation,
        COUNT(*) as num_operacoes
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
    GROUP BY DATE(committed_at), operation
    ORDER BY data DESC, operation
""").show()

# 6. Análise de distribuição de arquivos por partição
print("\n6️⃣ Análise de Distribuição de Arquivos por Partição:")

spark.sql("""
    SELECT 
        partition.data_venda_year as ano,
        partition.categoria,
        COUNT(*) as num_arquivos,
        ROUND(SUM(file_size_in_bytes) / 1024.0 / 1024.0, 2) as tamanho_mb,
        ROUND(AVG(file_size_in_bytes) / 1024.0, 2) as tamanho_medio_kb,
        SUM(record_count) as total_registros
    FROM hadoop_catalog.default.vendas_ecommerce.files
    GROUP BY partition.data_venda_year, partition.categoria
    ORDER BY ano DESC, tamanho_mb DESC
""").show()

# 7. Relatório final consolidado em formato de tabela
print("\n7️⃣ Relatório Final Consolidado:")

spark.sql("""
    SELECT 
        'Snapshots' as tipo_metadado,
        CAST(COUNT(*) AS STRING) as quantidade,
        'Operações na tabela' as descricao
    FROM hadoop_catalog.default.vendas_ecommerce.snapshots
    
    UNION ALL
    
    SELECT 
        'Arquivos' as tipo_metadado,
        CAST(COUNT(*) AS STRING) as quantidade,
        'Arquivos de dados' as descricao
    FROM hadoop_catalog.default.vendas_ecommerce.files
    
    UNION ALL
    
    SELECT 
        'Partições' as tipo_metadado,
        CAST(COUNT(DISTINCT CONCAT(CAST(partition.data_venda_year AS STRING), '-', partition.categoria)) AS STRING) as quantidade,
        'Partições únicas' as descricao
    FROM hadoop_catalog.default.vendas_ecommerce.files
    
    UNION ALL
    
    SELECT 
        'Tamanho Total (MB)' as tipo_metadado,
        CAST(ROUND(SUM(file_size_in_bytes) / 1024.0 / 1024.0, 2) AS STRING) as quantidade,
        'Armazenamento total' as descricao
    FROM hadoop_catalog.default.vendas_ecommerce.files
    
    UNION ALL
    
    SELECT 
        'Total de Registros' as tipo_metadado,
        CAST(SUM(record_count) AS STRING) as quantidade,
        'Registros nos arquivos' as descricao
    FROM hadoop_catalog.default.vendas_ecommerce.files
""").show(truncate=False)

print("\n✅ Análise avançada usando múltiplas tabelas de metadados concluída!")
print("=" * 70)

🎯 PONTUAÇÃO EXTRA 2: Análise avançada usando múltiplas tabelas de metadados

📊 Combinando dados de snapshots, files e partitions para análise completa:

📊 Relatório de Performance e Otimização Baseado em Metadados:

1️⃣ Análise de Snapshots:
+---------+-------------+--------------------+--------------------+
|operation|num_snapshots|   primeira_operacao|     ultima_operacao|
+---------+-------------+--------------------+--------------------+
|overwrite|            2|2025-11-10 05:02:...|2025-11-10 05:02:...|
|   append|            1|2025-11-10 05:02:...|2025-11-10 05:02:...|
+---------+-------------+--------------------+--------------------+


2️⃣ Análise de Files:
+--------------+-------------------+----------------+-------------------+---------------+---------------------------+
|total_arquivos|tamanho_total_bytes|tamanho_total_mb|tamanho_medio_bytes|total_registros|registros_medio_por_arquivo|
+--------------+-------------------+----------------+-------------------+---------------+-

In [22]:
# ============================================
# PONTUAÇÃO EXTRA 3: Integração com PostgreSQL (tabela customers)
# ============================================

print("=" * 70)
print("🎯 PONTUAÇÃO EXTRA 3: Integração com PostgreSQL (tabela customers)")
print("=" * 70)

# Configuração da conexão JDBC com PostgreSQL
print("\n📊 Configurando conexão com PostgreSQL...")

# Credenciais do PostgreSQL (conforme docker-compose.yml)
jdbc_hostname = "postgres-erp"  # Nome do serviço no docker-compose (rede Docker)
jdbc_port = 5432
jdbc_database = "northwind"
jdbc_username = "postgres"
jdbc_password = "postgres"

# URL JDBC de conexão
jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"

# Propriedades de conexão JDBC
connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "org.postgresql.Driver"
}

print(f"✅ Configuração JDBC:")
print(f"   URL: {jdbc_url}")
print(f"   Database: {jdbc_database}")
print(f"   User: {jdbc_username}")

# Função para ler dados do PostgreSQL
def read_postgres_customers():
    """Lê dados da tabela customers do PostgreSQL"""
    query = """
        (SELECT 
            customer_id,
            company_name,
            contact_name,
            contact_title,
            address,
            city,
            region,
            postal_code,
            country,
            phone,
            fax
         FROM customers
        ) AS customers
    """
    df = spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=connection_properties
    )
    return df

# Ler dados do PostgreSQL
print("\n🔄 Lendo dados da tabela customers do PostgreSQL...")
try:
    df_postgres_customers = read_postgres_customers()
    total_customers = df_postgres_customers.count()
    print(f"✅ Dados lidos do PostgreSQL com sucesso!")
    print(f"📊 Total de registros: {total_customers}")
    print("\n📊 Amostra dos dados do PostgreSQL:")
    df_postgres_customers.show(5, truncate=False)
except Exception as e:
    print(f"⚠️ Erro ao conectar ao PostgreSQL: {e}")
    print("\n💡 Tentando diagnóstico...")
    print("💡 Verifique se:")
    print("   1. O container postgres-erp está rodando")
    print("   2. Ambos os containers estão na mesma rede Docker")
    print("   3. O driver JDBC está disponível")
    print("\n💡 Para verificar, execute no terminal:")
    print("   docker ps | grep postgres")
    print("   docker network inspect iceberg-pyspark-mba_plataform-network")
    raise

# Criar view temporária com dados do PostgreSQL
df_postgres_customers.createOrReplaceTempView("customers_postgres")

# Integrar dados transacionais (PostgreSQL) com dados analíticos (Iceberg)
print("\n📊 Integrando dados transacionais (PostgreSQL) com dados analíticos (Iceberg):")

# 1. Análise dos clientes do PostgreSQL
print("\n1️⃣ Análise dos clientes do PostgreSQL:")

spark.sql("""
    SELECT 
        country,
        COUNT(*) as total_clientes,
        COUNT(DISTINCT city) as cidades_unicas
    FROM customers_postgres
    GROUP BY country
    ORDER BY total_clientes DESC
    LIMIT 10
""").show(truncate=False)

# 2. Verificar correspondência entre customer_id do PostgreSQL e cliente_id das vendas
print("\n2️⃣ Verificando correspondência entre customer_id (PostgreSQL) e cliente_id (Iceberg):")

# Verificar se há alguma correspondência
correspondencia = spark.sql("""
    SELECT 
        c.customer_id as customer_id_pg,
        v.cliente_id as cliente_id_iceberg,
        COUNT(*) as matches
    FROM customers_postgres c
    INNER JOIN hadoop_catalog.default.vendas_ecommerce v
        ON c.customer_id = v.cliente_id
    GROUP BY c.customer_id, v.cliente_id
    LIMIT 5
""").count()

if correspondencia > 0:
    print(f"✅ Encontradas {correspondencia} correspondências!")
    
    # JOIN real entre PostgreSQL e Iceberg
    print("\n📊 JOIN entre PostgreSQL e Iceberg:")
    spark.sql("""
        SELECT 
            c.customer_id,
            c.company_name,
            c.city,
            c.country,
            COUNT(v.venda_id) as total_vendas,
            ROUND(SUM(v.preco_unitario * v.quantidade), 2) as receita_total,
            ROUND(AVG(v.preco_unitario * v.quantidade), 2) as ticket_medio
        FROM customers_postgres c
        INNER JOIN hadoop_catalog.default.vendas_ecommerce v
            ON c.customer_id = v.cliente_id
        GROUP BY c.customer_id, c.company_name, c.city, c.country
        ORDER BY receita_total DESC
        LIMIT 10
    """).show(truncate=False)
else:
    print("⚠️ Não há correspondência direta entre customer_id do PostgreSQL e cliente_id das vendas")
    print("💡 Os customer_id do PostgreSQL são diferentes (ex: ALFKI, ANATR)")
    print("💡 Os cliente_id das vendas são diferentes (ex: CLI001, CLI002)")
    print("💡 Vamos demonstrar a integração criando dados de exemplo")
    
    # Criar alguns registros de vendas com customer_id do PostgreSQL para demonstração
    print("\n🔄 Criando registros de exemplo para demonstração...")
    
    # Pegar alguns customer_id do PostgreSQL
    sample_customers = spark.sql("SELECT customer_id FROM customers_postgres LIMIT 5").collect()
    customer_ids = [row['customer_id'] for row in sample_customers]
    
    print(f"📊 Usando customer_id do PostgreSQL: {customer_ids}")
    
    # Inserir vendas de exemplo com esses customer_id
    for i, cust_id in enumerate(customer_ids, start=1000000):
        spark.sql(f"""
            INSERT INTO hadoop_catalog.default.vendas_ecommerce
            VALUES (
                {i},
                'Produto Demo',
                'Eletrônicos',
                1,
                100.00,
                CURRENT_DATE(),
                '{cust_id}',
                101,
                0.0,
                'online'
            )
        """)
    
    print(f"✅ Criados {len(customer_ids)} registros de exemplo com customer_id do PostgreSQL")
    
    # Agora fazer o JOIN
    print("\n📊 JOIN entre PostgreSQL e Iceberg (com dados de exemplo):")
    spark.sql("""
        SELECT 
            c.customer_id,
            c.company_name,
            c.city,
            c.country,
            COUNT(v.venda_id) as total_vendas,
            ROUND(SUM(v.preco_unitario * v.quantidade), 2) as receita_total,
            ROUND(AVG(v.preco_unitario * v.quantidade), 2) as ticket_medio
        FROM customers_postgres c
        INNER JOIN hadoop_catalog.default.vendas_ecommerce v
            ON c.customer_id = v.cliente_id
        GROUP BY c.customer_id, c.company_name, c.city, c.country
        ORDER BY receita_total DESC
    """).show(truncate=False)

# 3. Análise de clientes do PostgreSQL por país
print("\n3️⃣ Análise de clientes do PostgreSQL por país:")

spark.sql("""
    SELECT 
        country,
        region,
        COUNT(*) as total_clientes,
        COUNT(DISTINCT city) as cidades_unicas
    FROM customers_postgres
    GROUP BY country, region
    ORDER BY total_clientes DESC
    LIMIT 10
""").show(truncate=False)

# 4. Demonstração de integração: dados do PostgreSQL + análise de vendas
print("\n4️⃣ Demonstração de integração: Dados do PostgreSQL + Análise de Vendas:")
print("💡 Mostrando estrutura de dados transacionais (PostgreSQL) e analíticos (Iceberg)")

print("\n📊 Dados Transacionais (PostgreSQL):")
spark.sql("SELECT customer_id, company_name, city, country FROM customers_postgres LIMIT 5").show(truncate=False)

print("\n📊 Dados Analíticos (Iceberg):")
spark.sql("SELECT cliente_id, categoria, COUNT(*) as vendas FROM hadoop_catalog.default.vendas_ecommerce GROUP BY cliente_id, categoria LIMIT 5").show(truncate=False)

# 5. Análise combinada: clientes do PostgreSQL + vendas por categoria
print("\n5️⃣ Análise combinada: Clientes do PostgreSQL + Vendas por Categoria:")

spark.sql("""
    SELECT 
        c.country,
        v.categoria,
        COUNT(DISTINCT c.customer_id) as clientes_unicos,
        COUNT(v.venda_id) as total_vendas,
        ROUND(SUM(v.preco_unitario * v.quantidade), 2) as receita_total
    FROM customers_postgres c
    INNER JOIN hadoop_catalog.default.vendas_ecommerce v
        ON c.customer_id = v.cliente_id
    GROUP BY c.country, v.categoria
    ORDER BY receita_total DESC
    LIMIT 10
""").show(truncate=False)

print("\n✅ Integração com dados do PostgreSQL demonstrada!")
print("\n💡 Dados transacionais (PostgreSQL) integrados com dados analíticos (Iceberg)")
print("💡 JOIN entre tabelas de diferentes sistemas funcionando corretamente!")
print("=" * 70)

🎯 PONTUAÇÃO EXTRA 3: Integração com PostgreSQL (tabela customers)

📊 Configurando conexão com PostgreSQL...
✅ Configuração JDBC:
   URL: jdbc:postgresql://postgres-erp:5432/northwind
   Database: northwind
   User: postgres

🔄 Lendo dados da tabela customers do PostgreSQL...
✅ Dados lidos do PostgreSQL com sucesso!
📊 Total de registros: 91

📊 Amostra dos dados do PostgreSQL:
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|company_name                      |contact_name      |contact_title       |address                      |city       |region|postal_code|country|phone         |fax           |
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|ALFKI      |Alfreds Futterkiste               |Maria Ande